# Computing NDVI from Landsat

**Series:** NDVI and Remote Sensing (notebook 1 of 3)
**Time:** about 25 minutes

**In this notebook you will:**
- Learn why healthy plants look bright in near infrared
- Open satellite bands with `rasterio`
- Compute NDVI with one line of NumPy
- Map NDVI, and optionally save it as a GeoTIFF for QGIS

**You should already know:** basic Python. No remote sensing background needed.

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## What is NDVI?

Satellites measure how much light each patch of ground reflects in several **bands** (colors),
including some our eyes cannot see.

- Healthy leaves **absorb red light** (they use it for photosynthesis).
- Healthy leaves **strongly reflect near infrared (NIR)** light.

The **Normalized Difference Vegetation Index** compares the two:

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

| NDVI | Typical surface |
|---|---|
| below 0 | Water, snow, clouds |
| 0 to 0.2 | Bare soil, buildings, roads |
| 0.2 to 0.5 | Grass, shrubs, crops early in the season |
| above 0.5 | Dense, healthy vegetation |

For Landsat 8 and 9, **Band 4 is red** and **Band 5 is NIR**.

## Step 1: Install and import

We do not download the satellite files. `rasterio` opens them directly from their web address
and reads the pixels into memory.

In [ ]:
!pip install -q rasterio

import numpy as np
import matplotlib.pyplot as plt
import rasterio

# rasterio reads the bands straight from GitHub into memory; nothing is downloaded to disk
RED_URL = "https://raw.githubusercontent.com/SpatialTurn/DataAnalysis/main/data/landsat/band4.TIF"   # Band 4: red
NIR_URL = "https://raw.githubusercontent.com/SpatialTurn/DataAnalysis/main/data/landsat/band5.TIF"   # Band 5: near infrared (NIR)

## Step 2: Open the red band and look at it

A satellite band is a **raster**: a grid of numbers, one per 30 m by 30 m pixel.

In [ ]:
with rasterio.open(RED_URL) as src:
    print("Size (rows, columns):", src.height, src.width)
    print("Coordinate system:", src.crs)
    print("Pixel size (m):", src.res)
    red = src.read(1).astype("float32")
    profile = src.profile

with rasterio.open(NIR_URL) as src:
    nir = src.read(1).astype("float32")

Each band is about 90 MB, so this cell takes a few seconds.

The scene covers about 230 km by 235 km of Indiana. The edges of the file are filled with 0,
where the satellite did not see anything. We mark those as missing (`NaN`) so they do not affect
our results.

In [ ]:
red[red == 0] = np.nan
nir[nir == 0] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(red, cmap="Reds")
axes[0].set_title("Band 4: Red")
axes[1].imshow(nir, cmap="Greys_r")
axes[1].set_title("Band 5: Near infrared")
for ax in axes:
    ax.axis("off")
plt.show()

Fields and forests look **dark in red** but **bright in NIR**. That contrast is what NDVI measures.

## Step 3: Convert to reflectance

These are **Landsat Collection 2 Level-2** (surface reflectance) bands. To save space, USGS stores
reflectance as whole numbers. The documented conversion is:

`reflectance = stored value * 0.0000275 - 0.2`

Reflectance is the share of light reflected, from 0 (none) to 1 (all). We clip to that range
because a few very dark pixels (deep water, shadows) come out slightly below 0.

In [ ]:
red = np.clip(red * 0.0000275 - 0.2, 0, 1)
nir = np.clip(nir * 0.0000275 - 0.2, 0, 1)

print("Median red reflectance:", round(float(np.nanmedian(red)), 3))
print("Median NIR reflectance:", round(float(np.nanmedian(nir)), 3))

> If you download your own scenes from [USGS EarthExplorer](https://earthexplorer.usgs.gov/), choose
> **Landsat 8-9 OLI/TIRS C2 L2** and use the `SR_B4` and `SR_B5` files with the same conversion.

## Step 4: Compute NDVI

In [ ]:
ndvi = (nir - red) / (nir + red)

print("Min: ", round(float(np.nanmin(ndvi)), 3))
print("Mean:", round(float(np.nanmean(ndvi)), 3))
print("Max: ", round(float(np.nanmax(ndvi)), 3))

NumPy does the math for all 60 million pixels at once. No loop needed.

## Step 5: Map NDVI

In [ ]:
plt.figure(figsize=(9, 8))
plt.imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
plt.colorbar(label="NDVI")
plt.title("NDVI, Landsat scene over Indiana")
plt.axis("off")
plt.show()

Green is more vegetation, red is less. Cities and water stand out in red and orange.

## Step 6: Look at the distribution

In [ ]:
plt.figure(figsize=(8, 3))
plt.hist(ndvi[~np.isnan(ndvi)], bins=100, color="green")
plt.xlabel("NDVI")
plt.ylabel("Number of pixels")
plt.title("How NDVI values are spread")
plt.show()

## Step 7 (optional): Save NDVI to open in QGIS

Everything so far lives in memory. If you want to open the result in QGIS, save it as a GeoTIFF. We copy the location information (`profile`) from the red band, so the NDVI file lines up
exactly in QGIS or any GIS.

In [ ]:
profile.update(dtype="float32", count=1, nodata=np.nan, compress="lzw")
with rasterio.open("ndvi.tif", "w", **profile) as dst:
    dst.write(ndvi.astype("float32"), 1)
print("Saved ndvi.tif")

In Colab, open the **Files** panel on the left, right click `ndvi.tif` and choose **Download**.

## Try it yourself

Zoom in: show only the pixels in rows 3000 to 4000 and columns 3000 to 4000 of the NDVI array. What kind of landscape do you see?

*Hint:* arrays are sliced as `array[row_start:row_end, col_start:col_end]`.

<details>
<summary><b>Show a solution</b></summary>

```python
plt.imshow(ndvi[3000:4000, 3000:4000], cmap="RdYlGn", vmin=-0.2, vmax=0.6)
plt.colorbar(label="NDVI")
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- A satellite band is a raster: a grid of numbers with a location.
- NDVI = (NIR - Red) / (NIR + Red); for Landsat 8 and 9 that is bands 5 and 4.
- Mark no-data pixels as `NaN` before calculating.
- Reuse the input `profile` so the saved GeoTIFF lines up in a GIS.

**Next:** [Classifying NDVI](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/04b_Classifying_NDVI.ipynb)